# IMDb KNN Classification

This beginner-friendly project uses **K-Nearest Neighbors (KNN)** to classify IMDb movies as:

- `1` = High Rated (`IMDB_Rating >= 8.0`)
- `0` = Not High Rated (`IMDB_Rating < 8.0`)

The project follows the KNN concepts from the course notes: **K**, distance, `weights`, neighbor-search algorithms, scaling, classification metrics, and confusion matrix.

> Important: `IMDB_Rating` is used only to create the target. It is **not** included in the model features, because that would leak the answer to the model.

## 1. Import Libraries

In [ ]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay,
)


## 2. Load Dataset

In [ ]:
# Works whether Jupyter starts inside notebooks/ or at the project root.
data_path = Path("../data/imdb_top_1000.csv")

if not data_path.exists():
    data_path = Path("data/imdb_top_1000.csv")

df = pd.read_csv(data_path)

df.head()


In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())


In [ ]:
df.info()


In [ ]:
df.isnull().sum()


## 3. Data Cleaning

The numeric columns need a little cleaning:

- `Runtime`: remove `" min"`
- `Gross`: remove commas and convert to a number
- `Released_Year`: convert to numeric
- Missing `Meta_score` and `Gross`: fill with the median
- Missing `Certificate`: replace with `"Unknown"`

In [ ]:
df["Released_Year"] = pd.to_numeric(df["Released_Year"], errors="coerce")

df["Runtime"] = (
    df["Runtime"]
    .str.replace(" min", "", regex=False)
)
df["Runtime"] = pd.to_numeric(df["Runtime"], errors="coerce")

df["Gross"] = (
    df["Gross"]
    .str.replace(",", "", regex=False)
)
df["Gross"] = pd.to_numeric(df["Gross"], errors="coerce")

df["Meta_score"] = df["Meta_score"].fillna(df["Meta_score"].median())
df["Gross"] = df["Gross"].fillna(df["Gross"].median())
df["Certificate"] = df["Certificate"].fillna("Unknown")

# Only one Released_Year value is not numeric in this dataset.
df = df.dropna(subset=["Released_Year"]).reset_index(drop=True)


In [ ]:
df[
    ["Released_Year", "Runtime", "Meta_score", "Gross", "Certificate"]
].isnull().sum()


## 4. Create the Classification Target

In [ ]:
df["High_Rated"] = (df["IMDB_Rating"] >= 8.0).astype(int)

print(df["High_Rated"].value_counts())
print("\nPercentage:")
print((df["High_Rated"].value_counts(normalize=True) * 100).round(2))


## 5. Feature Engineering

We use the features we selected for this project:

### Numeric
- Released Year
- Runtime
- Meta Score
- Number of Votes
- Gross Revenue

### Categorical
- Genre
- Certificate

`No_of_Votes` and `Gross` are strongly right-skewed, so `log1p` is used before scaling.  
This prevents very large values from dominating KNN distance calculations.

In [ ]:
df["No_of_Votes_log"] = np.log1p(df["No_of_Votes"])
df["Gross_log"] = np.log1p(df["Gross"])

numeric_features = [
    "Released_Year",
    "Runtime",
    "Meta_score",
    "No_of_Votes_log",
    "Gross_log",
]

X_numeric = df[numeric_features].copy()

genre_encoded = df["Genre"].str.get_dummies(sep=", ").astype(int)

certificate_encoded = pd.get_dummies(
    df["Certificate"],
    prefix="Certificate",
    dtype=int,
)

X_categorical = pd.concat(
    [genre_encoded, certificate_encoded],
    axis=1,
)

y = df["High_Rated"].copy()

print("Numeric features:", X_numeric.shape[1])
print("Categorical features:", X_categorical.shape[1])
print("Total features:", X_numeric.shape[1] + X_categorical.shape[1])


## 6. Train / Validation / Test Split

We keep a separate **validation set** for choosing KNN settings.

- Train: 60%
- Validation: 20%
- Test: 20%

The test set is not used when choosing `K`, `weights`, or distance metric.

In [ ]:
all_indices = np.arange(len(df))

train_idx, temp_idx = train_test_split(
    all_indices,
    test_size=0.40,
    random_state=42,
    stratify=y,
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42,
    stratify=y.iloc[temp_idx],
)

print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))


## 7. Scale Only Numeric Features

KNN is distance-based, so scaling is important.

We fit `StandardScaler` **only on the training numeric data**.  
The one-hot encoded Genre and Certificate columns stay as `0/1`.

In [ ]:
scaler = StandardScaler()

X_train_num = pd.DataFrame(
    scaler.fit_transform(X_numeric.iloc[train_idx]),
    columns=numeric_features,
)

X_val_num = pd.DataFrame(
    scaler.transform(X_numeric.iloc[val_idx]),
    columns=numeric_features,
)

X_test_num = pd.DataFrame(
    scaler.transform(X_numeric.iloc[test_idx]),
    columns=numeric_features,
)

X_train_cat = X_categorical.iloc[train_idx].reset_index(drop=True)
X_val_cat = X_categorical.iloc[val_idx].reset_index(drop=True)
X_test_cat = X_categorical.iloc[test_idx].reset_index(drop=True)

X_train = pd.concat([X_train_num, X_train_cat], axis=1)
X_val = pd.concat([X_val_num, X_val_cat], axis=1)
X_test = pd.concat([X_test_num, X_test_cat], axis=1)

y_train = y.iloc[train_idx].reset_index(drop=True)
y_val = y.iloc[val_idx].reset_index(drop=True)
y_test = y.iloc[test_idx].reset_index(drop=True)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)


## 8. Baseline KNN Model

In [ ]:
baseline_model = KNeighborsClassifier(
    n_neighbors=5,
    weights="uniform",
    algorithm="auto",
    p=2,  # Euclidean distance
)

baseline_model.fit(X_train, y_train)

baseline_pred = baseline_model.predict(X_val)

print("Baseline Validation Metrics")
print("Accuracy :", round(accuracy_score(y_val, baseline_pred), 3))
print("Precision:", round(precision_score(y_val, baseline_pred), 3))
print("Recall   :", round(recall_score(y_val, baseline_pred), 3))
print("F1 Score :", round(f1_score(y_val, baseline_pred), 3))


## 9. Tune K, Weights, and Distance

Instead of choosing K only by Accuracy, this project selects the settings using **F1 score for class 1 (High Rated)**.

We compare:

- `K = 1, 3, 5, ..., 31`
- `weights = uniform / distance`
- `p = 1` → Manhattan distance
- `p = 2` → Euclidean distance

In [ ]:
results = []

for k in range(1, 32, 2):
    for weight in ["uniform", "distance"]:
        for p_value in [1, 2]:

            model = KNeighborsClassifier(
                n_neighbors=k,
                weights=weight,
                algorithm="auto",
                p=p_value,
            )

            model.fit(X_train, y_train)
            pred = model.predict(X_val)

            results.append(
                {
                    "k": k,
                    "weights": weight,
                    "p": p_value,
                    "accuracy": accuracy_score(y_val, pred),
                    "precision": precision_score(y_val, pred, zero_division=0),
                    "recall": recall_score(y_val, pred, zero_division=0),
                    "f1": f1_score(y_val, pred, zero_division=0),
                }
            )

results_df = pd.DataFrame(results).sort_values(
    by=["f1", "accuracy"],
    ascending=False,
).reset_index(drop=True)

results_df.head(10)


### Best F1 for each K

In [ ]:
best_f1_by_k = results_df.groupby("k")["f1"].max().sort_index()

plt.figure(figsize=(8, 5))
plt.plot(
    best_f1_by_k.index,
    best_f1_by_k.values,
    marker="o",
)
plt.xlabel("K")
plt.ylabel("Best Validation F1")
plt.title("K Value vs Best F1 Score")
plt.grid(alpha=0.3)
plt.show()


### Best Hyperparameters

In [ ]:
best_row = results_df.iloc[0]

best_k = int(best_row["k"])
best_weight = best_row["weights"]
best_p = int(best_row["p"])

print("Best K:", best_k)
print("Best weights:", best_weight)
print(
    "Best distance:",
    "Manhattan" if best_p == 1 else "Euclidean",
)
print("Validation F1:", round(best_row["f1"], 3))
print("Validation Accuracy:", round(best_row["accuracy"], 3))


## 10. Compare Neighbor Search Algorithms

In [ ]:
algorithm_results = []

for algorithm in ["auto", "brute", "kd_tree", "ball_tree"]:
    start = time.perf_counter()

    model = KNeighborsClassifier(
        n_neighbors=best_k,
        weights=best_weight,
        algorithm=algorithm,
        p=best_p,
    )

    model.fit(X_train, y_train)
    pred = model.predict(X_val)

    elapsed = time.perf_counter() - start

    algorithm_results.append(
        {
            "algorithm": algorithm,
            "f1": f1_score(y_val, pred, zero_division=0),
            "accuracy": accuracy_score(y_val, pred),
            "time_seconds": elapsed,
        }
    )

algorithm_results_df = pd.DataFrame(algorithm_results)
algorithm_results_df


The search algorithm mainly changes **how neighbors are found** and therefore often changes speed more than predictive quality.

## 11. Tuned KNN on the Validation Set

In [ ]:
tuned_model = KNeighborsClassifier(
    n_neighbors=best_k,
    weights=best_weight,
    algorithm="auto",
    p=best_p,
)

tuned_model.fit(X_train, y_train)
tuned_val_pred = tuned_model.predict(X_val)

print(classification_report(y_val, tuned_val_pred))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_val,
    tuned_val_pred,
)
plt.title("Validation Confusion Matrix")
plt.show()


## 12. Bonus: Tune the Decision Threshold for F1

KNN normally uses a 0.50 probability threshold for class 1.  
Because our main goal is a better balance between **Precision** and **Recall**, we can test a few thresholds on the validation set and select the one with the best F1.

This section is optional, but it is useful for understanding why F1 can improve even when Accuracy does not.

In [ ]:
val_prob = tuned_model.predict_proba(X_val)[:, 1]

threshold_results = []

for threshold in np.arange(0.30, 0.71, 0.02):
    threshold_pred = (val_prob >= threshold).astype(int)

    threshold_results.append(
        {
            "threshold": round(float(threshold), 2),
            "precision": precision_score(
                y_val,
                threshold_pred,
                zero_division=0,
            ),
            "recall": recall_score(
                y_val,
                threshold_pred,
                zero_division=0,
            ),
            "f1": f1_score(
                y_val,
                threshold_pred,
                zero_division=0,
            ),
        }
    )

threshold_df = pd.DataFrame(threshold_results).sort_values(
    "f1",
    ascending=False,
).reset_index(drop=True)

threshold_df.head(10)


In [ ]:
best_threshold = float(threshold_df.iloc[0]["threshold"])

print("Best validation threshold:", best_threshold)
print("Best validation F1:", round(threshold_df.iloc[0]["f1"], 3))


## 13. Final Model and Test Evaluation

After choosing the KNN settings and threshold using the validation set, we combine the Train and Validation samples and evaluate **once** on the untouched Test set.

In [ ]:
X_train_final = pd.concat(
    [X_train, X_val],
    axis=0,
    ignore_index=True,
)

y_train_final = pd.concat(
    [y_train, y_val],
    axis=0,
    ignore_index=True,
)

final_model = KNeighborsClassifier(
    n_neighbors=best_k,
    weights=best_weight,
    algorithm="auto",
    p=best_p,
)

final_model.fit(X_train_final, y_train_final)


### Default KNN Prediction (0.50 threshold)

In [ ]:
test_pred_default = final_model.predict(X_test)

print("Default-threshold Test Report")
print(classification_report(y_test, test_pred_default))

print("F1:", round(f1_score(y_test, test_pred_default), 3))


### F1-Tuned Threshold Prediction

In [ ]:
test_prob = final_model.predict_proba(X_test)[:, 1]
test_pred_tuned = (test_prob >= best_threshold).astype(int)

print("Tuned-threshold Test Report")
print(classification_report(y_test, test_pred_tuned))

print("Accuracy :", round(accuracy_score(y_test, test_pred_tuned), 3))
print("Precision:", round(precision_score(y_test, test_pred_tuned), 3))
print("Recall   :", round(recall_score(y_test, test_pred_tuned), 3))
print("F1 Score :", round(f1_score(y_test, test_pred_tuned), 3))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test,
    test_pred_tuned,
)
plt.title("Final Test Confusion Matrix")
plt.show()


## 14. Conclusion

This project applies KNN classification to IMDb movie data and practices:

- Data cleaning
- Feature selection and one-hot encoding
- Numeric feature scaling
- K-Nearest Neighbors classification
- Choosing K
- `uniform` vs `distance` weights
- Manhattan vs Euclidean distance
- `auto`, `brute`, `kd_tree`, and `ball_tree`
- Accuracy, Precision, Recall, and F1
- Confusion Matrix and Classification Report
- Optional decision-threshold tuning for F1

### Why this version is stronger

The original baseline scaled every feature, including one-hot categorical columns, and selected K mainly using Accuracy.  
This version:

1. Scales only numeric features.
2. Log-transforms highly skewed Votes and Gross values.
3. Uses a separate Validation set for model selection.
4. Chooses KNN settings based on F1 for the High-Rated class.
5. Keeps the Test set untouched until the final evaluation.